# Fraud Detection EDA + DVC Decision Notebook

## Design and implement a ML system for detecting fraud transactions with real-world dataset

### Notebook scope

Notebook này tập trung vào 3 mục tiêu hiện tại của project:

1. **EDA thật kỹ, thật thực chiến** cho fraud detection dataset  
2. **Đánh giá có nên dùng DVC hay không** với **repository hiện tại**  
3. Nếu nên, đưa ra **cách triển khai DVC ở mức đủ dùng cho project sinh viên theo hướng MLOps**

### Business framing

Đây là bài toán **binary classification** để phát hiện giao dịch gian lận.

Trong fraud detection, accuracy gần như không đủ để đánh giá chất lượng hệ thống vì dữ liệu thường **rất mất cân bằng**.  
Những metric nhóm đang quan tâm và notebook này sẽ liên hệ trực tiếp tới là:

- **Recall**
- **Precision**
- **F1-score**
- **PR-AUC**

### EDA goals trong notebook này

Notebook sẽ ưu tiên kiểm tra các điểm quan trọng nhất cho fraud detection:

- class imbalance / fraud ratio
- missing values và sparsity
- duplicate rows / duplicate transaction IDs
- phân phối feature
- outliers
- khác biệt giữa fraud vs non-fraud
- correlation giữa các nhóm feature
- tín hiệu theo thời gian nếu có cột time
- các dấu hiệu có thể ảnh hưởng trực tiếp tới Recall, Precision, F1-score và PR-AUC ở bước modeling sau

### Repo-specific assumptions

Notebook này được viết để chạy với cấu trúc repo hiện tại của bạn, trong đó dữ liệu nằm trong thư mục `raw_data/`.

Theo tên file trong repo, dataset nhiều khả năng là **IEEE-CIS Fraud Detection style dataset** với hai phần chính ở train:

- `train_transaction.csv.gz`
- `train_identity.csv`

Notebook sẽ **không hard-code schema hoàn toàn**, mà sẽ:

- tự tìm repo root
- tự đọc header
- tự kiểm tra target
- tự chọn feature để phân tích theo mức độ tín hiệu và tính thực dụng

### Khi nào bạn cần sửa tay?

Bạn chỉ cần sửa tay nếu:

- đường dẫn thư mục dữ liệu không còn là `raw_data/`
- tên target không phải `isFraud`
- dataset thực tế khác format chuẩn đang giả định từ repo hiện tại


In [ ]:

from pathlib import Path
import io
import os
import re
import json
import math
import shutil
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
TARGET_CANDIDATES = ["isFraud", "fraud", "target", "Class", "class", "label"]

# Sampling config cho các bước nặng về visualization / correlation
PLOT_SAMPLE_NON_FRAUD = 25_000
PLOT_SAMPLE_FRAUD = 20_000
RANKING_SAMPLE_NON_FRAUD = 100_000
RANKING_SAMPLE_FRAUD = 20_000

MAX_NUMERIC_UNIVARIATE = 10
MAX_CATEGORICAL_UNIVARIATE = 8
MAX_HEATMAP_FEATURES = 24
MIN_CATEGORY_COUNT = 200

# Debug option: set số dòng nếu muốn test nhanh trên máy yếu
LIMIT_ROWS = None  # ví dụ: 100_000


## 1) Setup và đọc dữ liệu

Cell dưới đây sẽ:

- tự tìm `repo_root`
- kiểm tra sự tồn tại của các file trong `raw_data/`
- hiển thị kích thước file để audit nhanh dataset


In [ ]:

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "raw_data").exists():
            return path
    raise FileNotFoundError(
        "Không tìm thấy thư mục 'raw_data/'. "
        "Hãy chạy notebook từ repo root hoặc từ một thư mục con bên trong repo."
    )

def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan

repo_root = find_repo_root()
raw_data_dir = repo_root / "raw_data"

TRAIN_TRANSACTION_PATH = raw_data_dir / "train_transaction.csv.gz"
TRAIN_IDENTITY_PATH = raw_data_dir / "train_identity.csv"
TEST_TRANSACTION_PATH = raw_data_dir / "test_transaction.csv.gz"
TEST_IDENTITY_PATH = raw_data_dir / "test_identity.csv"
SAMPLE_SUBMISSION_PATH = raw_data_dir / "sample_submission.csv"

file_audit = pd.DataFrame({
    "file_name": [
        TRAIN_TRANSACTION_PATH.name,
        TRAIN_IDENTITY_PATH.name,
        TEST_TRANSACTION_PATH.name,
        TEST_IDENTITY_PATH.name,
        SAMPLE_SUBMISSION_PATH.name,
    ],
    "exists": [
        TRAIN_TRANSACTION_PATH.exists(),
        TRAIN_IDENTITY_PATH.exists(),
        TEST_TRANSACTION_PATH.exists(),
        TEST_IDENTITY_PATH.exists(),
        SAMPLE_SUBMISSION_PATH.exists(),
    ],
    "size_mb": [
        file_size_mb(TRAIN_TRANSACTION_PATH),
        file_size_mb(TRAIN_IDENTITY_PATH),
        file_size_mb(TEST_TRANSACTION_PATH),
        file_size_mb(TEST_IDENTITY_PATH),
        file_size_mb(SAMPLE_SUBMISSION_PATH),
    ],
    "path": [
        str(TRAIN_TRANSACTION_PATH),
        str(TRAIN_IDENTITY_PATH),
        str(TEST_TRANSACTION_PATH),
        str(TEST_IDENTITY_PATH),
        str(SAMPLE_SUBMISSION_PATH),
    ],
})

print(f"Repo root: {repo_root}")
display(file_audit)

missing_required = [
    path.name
    for path in [TRAIN_TRANSACTION_PATH, TRAIN_IDENTITY_PATH]
    if not path.exists()
]

if missing_required:
    raise FileNotFoundError(
        "Thiếu file train cần thiết để chạy notebook: "
        + ", ".join(missing_required)
    )

display(Markdown(
    f"""
**Ghi chú thực thi**

- Notebook này dùng **train set** cho EDA chính: `{TRAIN_TRANSACTION_PATH.name}` + `{TRAIN_IDENTITY_PATH.name}`
- Test files vẫn được audit để phục vụ phần **đánh giá DVC** và kiểm tra cấu trúc dữ liệu tổng thể
"""
))


### 1.1) Helper functions để load IEEE-CIS style data theo hướng tiết kiệm bộ nhớ

Repo hiện tại dùng file transaction ở dạng `.csv.gz`, nên cell này sẽ:

- đọc **header trước**
- tạo `dtype map` theo pattern tên cột
- load dữ liệu với `dtype` hợp lý để giảm memory footprint
- merge transaction + identity theo `TransactionID`

> Mục tiêu là để notebook vẫn thực dụng trên máy sinh viên, thay vì load full data hoàn toàn theo mặc định `float64/object`.


In [ ]:

CATEGORICAL_ID_COLS = {
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29", "id_30",
    "id_31", "id_33", "id_34", "id_35", "id_36", "id_37", "id_38"
}

CATEGORICAL_TRANSACTION_COLS = {
    "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
    *{f"M{i}" for i in range(1, 10)}
}

CATEGORICAL_IDENTITY_EXTRA = {"DeviceType", "DeviceInfo"}

def build_ieee_dtype_map(columns: list[str]) -> dict[str, str]:
    dtype_map: dict[str, str] = {}
    for col in columns:
        if col == "TransactionID":
            dtype_map[col] = "int32"
        elif col == "isFraud":
            dtype_map[col] = "int8"
        elif col == "TransactionDT":
            dtype_map[col] = "int32"
        elif col == "TransactionAmt":
            dtype_map[col] = "float32"
        elif col in CATEGORICAL_TRANSACTION_COLS or col in CATEGORICAL_ID_COLS or col in CATEGORICAL_IDENTITY_EXTRA:
            dtype_map[col] = "category"
        elif col.startswith(("V", "C", "D", "id_", "card", "addr", "dist")):
            dtype_map[col] = "float32"
        else:
            # fallback thực dụng cho schema IEEE-CIS style
            dtype_map[col] = "float32"
    return dtype_map

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    for col in df.columns:
        col_dtype = df[col].dtype
        try:
            if pd.api.types.is_integer_dtype(col_dtype):
                df[col] = pd.to_numeric(df[col], downcast="integer")
            elif pd.api.types.is_float_dtype(col_dtype):
                df[col] = pd.to_numeric(df[col], downcast="float")
            elif pd.api.types.is_object_dtype(col_dtype):
                nunique_ratio = df[col].nunique(dropna=True) / max(len(df), 1)
                if nunique_ratio < 0.5:
                    df[col] = df[col].astype("category")
        except Exception:
            pass
    end_mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"Memory usage: {start_mem:,.2f} MB -> {end_mem:,.2f} MB")
    return df

def read_header(path: Path, compression: str = "infer") -> list[str]:
    return pd.read_csv(path, nrows=0, compression=compression).columns.tolist()

def load_dataframe(path: Path, dtype_map: dict[str, str] | None = None, compression: str = "infer", nrows: int | None = None) -> pd.DataFrame:
    try:
        return pd.read_csv(
            path,
            dtype=dtype_map,
            compression=compression,
            low_memory=False,
            nrows=nrows
        )
    except Exception as e:
        print(f"Typed load failed for {path.name}: {e}")
        print("Falling back to default dtype inference...")
        return pd.read_csv(
            path,
            compression=compression,
            low_memory=False,
            nrows=nrows
        )

def dataframe_memory_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)

def infer_target_column(df: pd.DataFrame, candidates: list[str] | None = None) -> str | None:
    candidates = TARGET_CANDIDATES if candidates is None else candidates
    for col in candidates:
        if col in df.columns:
            return col
    return None

def infer_positive_label(series: pd.Series):
    values = sorted([x for x in series.dropna().unique().tolist()])
    if 1 in values:
        return 1
    if len(values) == 2:
        return values[-1]
    return None

def dedupe_keep_order(items):
    seen = set()
    output = []
    for item in items:
        if item not in seen:
            seen.add(item)
            output.append(item)
    return output


In [ ]:

transaction_columns = read_header(TRAIN_TRANSACTION_PATH, compression="gzip")
identity_columns = read_header(TRAIN_IDENTITY_PATH)

transaction_dtypes = build_ieee_dtype_map(transaction_columns)
identity_dtypes = build_ieee_dtype_map(identity_columns)

schema_preview = pd.DataFrame({
    "source": ["train_transaction"] * len(transaction_columns) + ["train_identity"] * len(identity_columns),
    "column": transaction_columns + identity_columns,
    "planned_dtype": [transaction_dtypes[c] for c in transaction_columns] + [identity_dtypes[c] for c in identity_columns],
})

print(f"train_transaction columns: {len(transaction_columns)}")
print(f"train_identity columns: {len(identity_columns)}")
display(schema_preview.head(30))
display(schema_preview.tail(30))


In [ ]:

train_transaction = load_dataframe(
    TRAIN_TRANSACTION_PATH,
    dtype_map=transaction_dtypes,
    compression="gzip",
    nrows=LIMIT_ROWS
)

train_identity = load_dataframe(
    TRAIN_IDENTITY_PATH,
    dtype_map=identity_dtypes,
    compression="infer",
    nrows=None if LIMIT_ROWS is None else min(LIMIT_ROWS, 200_000)
)

print("Before post-load optimization")
print(f"train_transaction memory: {dataframe_memory_mb(train_transaction):,.2f} MB")
print(f"train_identity memory: {dataframe_memory_mb(train_identity):,.2f} MB")

train_transaction = reduce_mem_usage(train_transaction)
train_identity = reduce_mem_usage(train_identity)

try:
    df = train_transaction.merge(
        train_identity,
        on="TransactionID",
        how="left",
        validate="one_to_one"
    )
except Exception as e:
    print(f"Merge validation warning: {e}")
    print("Falling back to merge without strict validation.")
    df = train_transaction.merge(
        train_identity,
        on="TransactionID",
        how="left"
    )

TARGET_COL = infer_target_column(df)
POSITIVE_LABEL = infer_positive_label(df[TARGET_COL]) if TARGET_COL is not None else None

print(f"Merged dataframe shape: {df.shape}")
print(f"Merged dataframe memory: {dataframe_memory_mb(df):,.2f} MB")
print(f"Target column detected: {TARGET_COL}")
print(f"Positive class inferred as: {POSITIVE_LABEL}")

if TARGET_COL is None:
    raise ValueError(
        "Không tìm thấy target column. "
        "Hãy chỉnh TARGET_CANDIDATES hoặc set TARGET_COL thủ công."
    )


## 2) Dataset Overview

Phần này sẽ trả lời các câu hỏi cơ bản nhưng bắt buộc phải rõ trước khi đi sâu:

- dataset sau khi merge có bao nhiêu dòng / cột?
- cột nào là target?
- kiểu dữ liệu đang ra sao?
- numerical vs categorical phân bố như thế nào?
- dữ liệu train có đúng format cho fraud detection hay không?


In [ ]:

print("Shape:", df.shape)
display(df.head())

dtype_counts = df.dtypes.astype(str).value_counts().rename_axis("dtype").reset_index(name="count")
display(dtype_counts)

overview_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing_pct": (df.isna().mean() * 100).values,
    "nunique": df.nunique(dropna=True).values,
}).sort_values(["missing_pct", "nunique"], ascending=[False, False])

display(overview_df.head(30))

buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

display(Markdown(
    f"""
**Quick overview**

- Số dòng sau merge: **{df.shape[0]:,}**
- Số cột sau merge: **{df.shape[1]:,}**
- Target detected: **`{TARGET_COL}`**
- Positive class: **`{POSITIVE_LABEL}`**
"""
))


## 3) Target Analysis

Đây là phần quan trọng nhất của fraud EDA.

Với fraud detection, class distribution ảnh hưởng trực tiếp tới:

- cách chia train/validation
- việc chọn threshold
- việc đánh giá bằng Recall / Precision / F1-score / PR-AUC
- cách đọc confusion matrix
- cách xử lý false positives vs false negatives

Nếu dữ liệu mất cân bằng mạnh, accuracy cao vẫn có thể là **ảo**.


In [ ]:

target_counts = df[TARGET_COL].value_counts(dropna=False).sort_index()
target_ratio = df[TARGET_COL].value_counts(normalize=True, dropna=False).sort_index()

target_summary = pd.DataFrame({
    "class_value": target_counts.index.astype(str),
    "count": target_counts.values,
    "ratio": target_ratio.values,
    "ratio_pct": target_ratio.values * 100,
})

display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(target_summary["class_value"], target_summary["count"])
axes[0].set_title("Target distribution (count)")
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel("Count")
for i, v in enumerate(target_summary["count"].values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10)

axes[1].bar(target_summary["class_value"], target_summary["ratio_pct"])
axes[1].set_title("Target distribution (percentage)")
axes[1].set_xlabel(TARGET_COL)
axes[1].set_ylabel("Percentage (%)")
for i, v in enumerate(target_summary["ratio_pct"].values):
    axes[1].text(i, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

fraud_ratio = float(target_ratio.loc[POSITIVE_LABEL]) if POSITIVE_LABEL in target_ratio.index else np.nan
majority_ratio = float(target_ratio.max())

display(Markdown(
    f"""
### Nhận xét target analysis

- Fraud ratio hiện tại là **{fraud_ratio:.4%}**.
- Class lớn nhất chiếm **{majority_ratio:.4%}** dữ liệu.
- Đây là dấu hiệu rất quan trọng cho bước modeling: accuracy có thể đánh lừa, trong khi **Recall, Precision, F1-score và đặc biệt là PR-AUC** mới phản ánh tốt hơn hiệu quả phát hiện fraud.
- Với dữ liệu mất cân bằng, threshold mặc định `0.5` thường **không tối ưu**.
"""
))


## 4) Basic Data Inspection trong EDA

Phần này chưa xây thành module Data Quality riêng, nhưng vẫn phải kiểm tra đủ kỹ để tránh EDA bị “đẹp giả”:

- Missing values
- Duplicate rows
- Duplicate `TransactionID`
- Constant / all-null columns
- Giá trị bất thường ở mức EDA
- Summary cho numerical và categorical features


In [ ]:

missing_df = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean() * 100).values,
    "dtype": df.dtypes.astype(str).values,
    "nunique": df.nunique(dropna=True).values,
}).sort_values("missing_pct", ascending=False)

duplicate_rows = int(df.duplicated().sum())
duplicate_transaction_ids = int(df.duplicated(subset=["TransactionID"]).sum()) if "TransactionID" in df.columns else np.nan
all_null_cols = missing_df.loc[missing_df["missing_pct"] == 100, "column"].tolist()
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
high_missing_cols = missing_df.loc[missing_df["missing_pct"] >= 90, "column"].tolist()

basic_checks = {
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "duplicate_rows": duplicate_rows,
    "duplicate_transaction_ids": duplicate_transaction_ids,
    "all_null_columns": len(all_null_cols),
    "constant_columns": len(constant_cols),
    "columns_missing_ge_90pct": len(high_missing_cols),
}

if "TransactionAmt" in df.columns:
    basic_checks["negative_transaction_amt"] = int((df["TransactionAmt"] < 0).sum())
    basic_checks["zero_transaction_amt"] = int((df["TransactionAmt"] == 0).sum())

display(pd.DataFrame([basic_checks]).T.rename(columns={0: "value"}))
display(missing_df.head(25))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_missing = missing_df.head(20).sort_values("missing_pct", ascending=True)
axes[0].barh(top_missing["column"], top_missing["missing_pct"])
axes[0].set_title("Top 20 columns by missing percentage")
axes[0].set_xlabel("Missing (%)")

axes[1].hist(missing_df["missing_pct"], bins=30)
axes[1].set_title("Distribution of missingness across columns")
axes[1].set_xlabel("Missing (%)")
axes[1].set_ylabel("Number of columns")

plt.tight_layout()
plt.show()

display(Markdown(
    f"""
### Nhận xét basic inspection

- Duplicate rows: **{duplicate_rows:,}**
- Duplicate `TransactionID`: **{duplicate_transaction_ids:,}**
- All-null columns: **{len(all_null_cols)}**
- Constant columns: **{len(constant_cols)}**
- Columns có missing >= 90%: **{len(high_missing_cols)}**

Trong fraud detection, missingness bản thân nó cũng có thể mang tín hiệu.  
Vì vậy ở bước này **không vội drop hàng loạt**, mà trước hết cần hiểu missing pattern và mối liên hệ của nó với fraud.
"""
))


In [ ]:

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols]

business_numeric_candidates = [
    "TransactionAmt", "TransactionDT",
    "card1", "card2", "card3", "card5",
    "addr1", "addr2", "dist1", "dist2",
    "C1", "C14", "D1", "D4", "D10", "D15",
    "id_01", "id_02", "id_17", "id_20", "id_32"
]
numeric_summary_cols = [c for c in business_numeric_candidates if c in df.columns and c != TARGET_COL][:15]

if numeric_summary_cols:
    numeric_summary = df[numeric_summary_cols].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
    display(numeric_summary)

categorical_summary_rows = []
categorical_summary_candidates = [
    "ProductCD", "card4", "card6",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo",
    "M1", "M4", "M6", "id_15", "id_28", "id_29", "id_30", "id_31",
]
categorical_summary_candidates = [c for c in categorical_summary_candidates if c in df.columns]

for col in categorical_summary_candidates:
    vc = df[col].astype("object").fillna("<MISSING>").value_counts(dropna=False)
    top_value = vc.index[0] if len(vc) else np.nan
    top_freq = int(vc.iloc[0]) if len(vc) else 0
    categorical_summary_rows.append({
        "column": col,
        "nunique": int(df[col].nunique(dropna=True)),
        "missing_pct": float(df[col].isna().mean() * 100),
        "top_value": top_value,
        "top_freq": top_freq,
    })

if categorical_summary_rows:
    categorical_summary = pd.DataFrame(categorical_summary_rows).sort_values(["nunique", "missing_pct"])
    display(categorical_summary)


## 5) Univariate Analysis

Với số lượng feature rất lớn, notebook này **không vẽ tất cả mọi cột** vì sẽ cực nặng và khó đọc.  
Thay vào đó, notebook chọn feature theo 2 tiêu chí:

1. **Business-important features** như `TransactionAmt`, `TransactionDT`, `card*`, `dist*`, `D*`, ...
2. **Signal-oriented features** có liên hệ mạnh hơn với target trên sample hợp lý

Mục tiêu là EDA đủ sâu nhưng vẫn chạy thực tế trên máy cá nhân.


In [ ]:

def stratified_sample(
    frame: pd.DataFrame,
    target_col: str,
    non_fraud_max: int | None = None,
    fraud_max: int | None = None,
    positive_label = 1,
    random_state: int = 42
) -> pd.DataFrame:
    if target_col not in frame.columns:
        return frame.copy()

    parts = []
    for label, group in frame.groupby(target_col, dropna=False):
        if label == positive_label:
            max_n = fraud_max
        else:
            max_n = non_fraud_max

        if max_n is None or len(group) <= max_n:
            parts.append(group)
        else:
            parts.append(group.sample(max_n, random_state=random_state))

    sampled = pd.concat(parts, axis=0)
    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return sampled

def choose_numeric_features(
    frame: pd.DataFrame,
    target_col: str,
    mandatory: list[str] | None = None,
    top_k: int = 10,
    min_non_null_ratio: float = 0.20
):
    mandatory = [] if mandatory is None else mandatory
    candidates = [
        c for c in frame.select_dtypes(include=[np.number]).columns
        if c not in {target_col, "TransactionID"}
        and frame[c].notna().mean() >= min_non_null_ratio
        and frame[c].nunique(dropna=True) > 5
    ]

    corr = frame[candidates].corrwith(frame[target_col]).abs().sort_values(ascending=False)
    corr = corr.dropna()

    selected = []
    for col in mandatory + corr.index.tolist():
        if col in candidates and col not in selected:
            selected.append(col)
        if len(selected) >= top_k:
            break

    return selected, corr

def is_categorical_like(series: pd.Series) -> bool:
    return (
        pd.api.types.is_categorical_dtype(series)
        or pd.api.types.is_object_dtype(series)
        or pd.api.types.is_string_dtype(series)
    )

def choose_categorical_features(
    frame: pd.DataFrame,
    target_col: str,
    mandatory: list[str] | None = None,
    top_k: int = 8,
    max_unique: int = 15,
    min_count: int = 200
):
    mandatory = [] if mandatory is None else mandatory
    cat_cols = [
        c for c in frame.columns
        if c not in {target_col, "TransactionID"} and is_categorical_like(frame[c])
    ]

    scored_rows = []
    for col in cat_cols:
        nunique = frame[col].nunique(dropna=True)
        if nunique == 0 or nunique > max_unique:
            continue

        stats = (
            frame[[col, target_col]]
            .assign(_cat=frame[col].astype("object").fillna("<MISSING>"))
            .groupby("_cat", dropna=False)[target_col]
            .agg(["count", "mean"])
        )
        stats = stats[stats["count"] >= min_count]
        if stats.empty:
            continue

        scored_rows.append({
            "feature": col,
            "nunique": int(nunique),
            "fraud_rate_range": float(stats["mean"].max() - stats["mean"].min()),
        })

    scored_df = pd.DataFrame(scored_rows).sort_values("fraud_rate_range", ascending=False) if scored_rows else pd.DataFrame(
        columns=["feature", "nunique", "fraud_rate_range"]
    )

    selected = []
    scored_features = scored_df["feature"].tolist() if not scored_df.empty else []
    for col in mandatory + scored_features:
        if col in cat_cols and col not in selected:
            selected.append(col)
        if len(selected) >= top_k:
            break

    return selected, scored_df

ranking_sample = stratified_sample(
    df,
    TARGET_COL,
    non_fraud_max=RANKING_SAMPLE_NON_FRAUD,
    fraud_max=RANKING_SAMPLE_FRAUD,
    positive_label=POSITIVE_LABEL,
    random_state=RANDOM_STATE
)

plot_sample = stratified_sample(
    df,
    TARGET_COL,
    non_fraud_max=PLOT_SAMPLE_NON_FRAUD,
    fraud_max=PLOT_SAMPLE_FRAUD,
    positive_label=POSITIVE_LABEL,
    random_state=RANDOM_STATE
)

mandatory_numeric = [
    "TransactionAmt", "TransactionDT",
    "card1", "card2", "card3", "card5",
    "addr1", "addr2", "dist1", "dist2",
    "C1", "C14", "D1", "D4", "D10", "D15",
    "id_01", "id_02", "id_17", "id_20", "id_32"
]

mandatory_categorical = [
    "ProductCD", "card4", "card6",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "M1", "M4", "M6",
    "id_15", "id_28", "id_29", "id_35", "id_36", "id_37", "id_38"
]

selected_numeric, numeric_corr = choose_numeric_features(
    ranking_sample,
    TARGET_COL,
    mandatory=[c for c in mandatory_numeric if c in ranking_sample.columns],
    top_k=MAX_NUMERIC_UNIVARIATE,
)

selected_categorical, categorical_signal = choose_categorical_features(
    ranking_sample,
    TARGET_COL,
    mandatory=[c for c in mandatory_categorical if c in ranking_sample.columns],
    top_k=MAX_CATEGORICAL_UNIVARIATE,
    max_unique=15,
    min_count=MIN_CATEGORY_COUNT,
)

print("Selected numeric features for univariate analysis:")
print(selected_numeric)

print("\nSelected categorical features for univariate analysis:")
print(selected_categorical)


In [ ]:

if selected_numeric:
    n_rows = len(selected_numeric)
    fig, axes = plt.subplots(n_rows, 2, figsize=(15, 4 * n_rows))
    if n_rows == 1:
        axes = np.array([axes])

    numeric_univariate_rows = []

    for i, col in enumerate(selected_numeric):
        s = df[col].dropna()
        plot_s = s.sample(min(len(s), 100_000), random_state=RANDOM_STATE) if len(s) > 100_000 else s

        axes[i, 0].hist(plot_s, bins=50)
        axes[i, 0].set_title(f"Histogram - {col}")
        axes[i, 0].set_xlabel(col)
        axes[i, 0].set_ylabel("Count")

        axes[i, 1].boxplot(plot_s, vert=False)
        axes[i, 1].set_title(f"Boxplot - {col}")
        axes[i, 1].set_xlabel(col)

        q01, q99 = s.quantile([0.01, 0.99]) if len(s) else (np.nan, np.nan)
        numeric_univariate_rows.append({
            "feature": col,
            "missing_pct": float(df[col].isna().mean() * 100),
            "mean": float(s.mean()) if len(s) else np.nan,
            "median": float(s.median()) if len(s) else np.nan,
            "std": float(s.std()) if len(s) else np.nan,
            "skew": float(s.skew()) if len(s) else np.nan,
            "p01": float(q01) if not pd.isna(q01) else np.nan,
            "p99": float(q99) if not pd.isna(q99) else np.nan,
            "max": float(s.max()) if len(s) else np.nan,
        })

    plt.tight_layout()
    plt.show()

    numeric_univariate_summary = pd.DataFrame(numeric_univariate_rows).sort_values("skew", key=lambda x: x.abs(), ascending=False)
    display(numeric_univariate_summary)

    display(Markdown(
        """
### Nhận xét numeric univariate

- Ưu tiên nhìn vào **skewness**, tail rất dài, và khoảng cách giữa `p99` với `max`.
- Với fraud detection, các tail cực dài và outlier không phải lúc nào cũng là noise; nhiều khi đó là **tín hiệu hành vi bất thường**.
- Các feature có missing cao nhưng vẫn tách class tốt vẫn đáng giữ lại cho modeling.
"""
    ))


In [ ]:

categorical_univariate_rows = []

for col in selected_categorical:
    plot_df = (
        df[[col, TARGET_COL]]
        .assign(_cat=df[col].astype("object").fillna("<MISSING>"))
        .copy()
    )

    top_order = plot_df["_cat"].value_counts().head(10).index.tolist()
    plot_df = plot_df[plot_df["_cat"].isin(top_order)].copy()

    stats = plot_df.groupby("_cat")[TARGET_COL].agg(["count", "mean"]).rename(columns={"mean": "fraud_rate"})
    stats["fraud_rate_pct"] = stats["fraud_rate"] * 100
    stats = stats.sort_values("count", ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    axes[0].barh(stats.index.tolist(), stats["count"].values)
    axes[0].set_title(f"Top categories by count - {col}")
    axes[0].set_xlabel("Count")
    axes[0].set_ylabel(col)
    axes[0].invert_yaxis()

    axes[1].barh(stats.index.tolist(), stats["fraud_rate_pct"].values)
    axes[1].set_title(f"Fraud rate by category - {col}")
    axes[1].set_xlabel("Fraud rate (%)")
    axes[1].set_ylabel(col)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    stats_reset = stats.reset_index().rename(columns={"_cat": "category"})
    for _, row in stats_reset.iterrows():
        categorical_univariate_rows.append({
            "feature": col,
            "category": row["category"],
            "count": int(row["count"]),
            "fraud_rate": float(row["fraud_rate"]),
            "fraud_rate_pct": float(row["fraud_rate_pct"]),
        })

if categorical_univariate_rows:
    categorical_univariate_summary = pd.DataFrame(categorical_univariate_rows)
    display(
        categorical_univariate_summary
        .sort_values(["fraud_rate", "count"], ascending=[False, False])
        .head(30)
    )

display(Markdown(
    """
### Nhận xét categorical univariate

- Với categorical features, không chỉ nhìn số lượng category mà cần nhìn **fraud rate theo category**.
- Category nào có **fraud rate vượt xa baseline fraud ratio** là ứng viên tốt cho modeling.
- Cần luôn đọc cùng với `count`, tránh kết luận từ category hiếm có rất ít mẫu.
"""
))


## 6) Fraud vs Non-Fraud Comparison

Đây là phần EDA có giá trị trực tiếp nhất cho modeling.

Mục tiêu:

- xem feature nào thật sự tách được fraud và non-fraud
- tránh chỉ nhìn phân phối chung
- ưu tiên insight nào có ích cho thresholding và trade-off Recall / Precision


In [ ]:

def build_numeric_separation_table(
    frame: pd.DataFrame,
    target_col: str,
    positive_label=1,
    min_non_null_ratio: float = 0.20
) -> pd.DataFrame:
    rows = []
    candidates = [
        c for c in frame.select_dtypes(include=[np.number]).columns
        if c not in {target_col, "TransactionID"}
        and frame[c].notna().mean() >= min_non_null_ratio
        and frame[c].nunique(dropna=True) > 5
    ]

    for col in candidates:
        valid = frame[[col, target_col]].dropna()
        if valid.empty:
            continue

        fraud_s = valid.loc[valid[target_col] == positive_label, col]
        non_fraud_s = valid.loc[valid[target_col] != positive_label, col]

        if len(fraud_s) < 30 or len(non_fraud_s) < 30:
            continue

        abs_corr = abs(valid[col].corr(valid[target_col])) if valid[col].nunique() > 1 else np.nan

        pooled_std = np.sqrt((fraud_s.var(ddof=1) + non_fraud_s.var(ddof=1)) / 2)
        effect_size = abs((fraud_s.mean() - non_fraud_s.mean()) / pooled_std) if pooled_std and not np.isnan(pooled_std) else np.nan

        q1, q3 = valid[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        robust_gap = abs(fraud_s.median() - non_fraud_s.median()) / iqr if iqr and not np.isnan(iqr) else np.nan

        rows.append({
            "feature": col,
            "missing_pct": float(frame[col].isna().mean() * 100),
            "fraud_mean": float(fraud_s.mean()),
            "non_fraud_mean": float(non_fraud_s.mean()),
            "fraud_median": float(fraud_s.median()),
            "non_fraud_median": float(non_fraud_s.median()),
            "abs_target_corr": float(abs_corr) if not pd.isna(abs_corr) else np.nan,
            "effect_size": float(effect_size) if not pd.isna(effect_size) else np.nan,
            "robust_gap_over_iqr": float(robust_gap) if not pd.isna(robust_gap) else np.nan,
        })

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    result = result.sort_values(
        ["effect_size", "abs_target_corr", "robust_gap_over_iqr"],
        ascending=False
    )
    return result

numeric_separation = build_numeric_separation_table(
    ranking_sample,
    TARGET_COL,
    positive_label=POSITIVE_LABEL
)

display(numeric_separation.head(20))

comparison_numeric_features = []
for col in (
    [c for c in ["TransactionAmt", "TransactionDT"] if c in df.columns]
    + (numeric_separation["feature"].head(8).tolist() if not numeric_separation.empty else [])
):
    if col not in comparison_numeric_features:
        comparison_numeric_features.append(col)
comparison_numeric_features = comparison_numeric_features[:6]

print("Comparison features:", comparison_numeric_features)


In [ ]:

for col in comparison_numeric_features:
    plot_df = plot_sample[[col, TARGET_COL]].dropna().copy()

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    fraud_series = plot_df.loc[plot_df[TARGET_COL] == POSITIVE_LABEL, col]
    non_fraud_series = plot_df.loc[plot_df[TARGET_COL] != POSITIVE_LABEL, col]

    axes[0].hist(non_fraud_series, bins=40, density=True, histtype="step", label="Non-Fraud")
    axes[0].hist(fraud_series, bins=40, density=True, histtype="step", label="Fraud")
    axes[0].set_title(f"Fraud vs Non-Fraud distribution - {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")
    axes[0].legend()

    axes[1].boxplot([non_fraud_series, fraud_series], labels=["Non-Fraud", "Fraud"])
    axes[1].set_title(f"Boxplot by class - {col}")
    axes[1].set_ylabel(col)

    plt.tight_layout()
    plt.show()

display(Markdown(
    """
### Nhận xét fraud vs non-fraud (numerical)

- Feature nào có phân phối giữa 2 class lệch nhau rõ hơn sẽ có tiềm năng đóng góp mạnh hơn cho classifier.
- Tuy nhiên, tách class tốt trên **một feature đơn lẻ** không có nghĩa là model cuối cùng chỉ cần feature đó.
- Fraud detection thường cần sự kết hợp giữa:
  - value magnitude
  - missingness pattern
  - categorical context
  - temporal context
  - interaction giữa nhiều feature
"""
))


In [ ]:

def build_categorical_fraud_rate_table(
    frame: pd.DataFrame,
    target_col: str,
    positive_label=1,
    min_count: int = 200,
    max_unique: int = 20
) -> pd.DataFrame:
    rows = []
    for col in frame.columns:
        if col in {target_col, "TransactionID"}:
            continue
        if not is_categorical_like(frame[col]):
            continue

        nunique = frame[col].nunique(dropna=True)
        if nunique == 0 or nunique > max_unique:
            continue

        work = frame[[col, target_col]].copy()
        work["_cat"] = work[col].astype("object").fillna("<MISSING>")
        stats = work.groupby("_cat")[target_col].agg(["count", "mean"]).reset_index()
        stats = stats[stats["count"] >= min_count]

        if stats.empty:
            continue

        for _, row in stats.iterrows():
            rows.append({
                "feature": col,
                "category": row["_cat"],
                "count": int(row["count"]),
                "fraud_rate": float(row["mean"]),
                "fraud_rate_pct": float(row["mean"] * 100),
            })

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    baseline = frame[target_col].mean()
    result["lift_vs_baseline"] = result["fraud_rate"] / baseline if baseline > 0 else np.nan
    return result.sort_values(["fraud_rate", "count"], ascending=[False, False])

categorical_fraud_rate_table = build_categorical_fraud_rate_table(
    ranking_sample,
    TARGET_COL,
    positive_label=POSITIVE_LABEL,
    min_count=MIN_CATEGORY_COUNT,
    max_unique=20
)

display(categorical_fraud_rate_table.head(30))

display(Markdown(
    """
### Nhận xét fraud vs non-fraud (categorical)

- Bảng trên giúp xác định category nào có fraud rate cao hơn nhiều so với baseline.
- Đây là tín hiệu rất hữu ích cho:
  - feature encoding ở bước sau
  - feature interaction
  - business rules / risk flags
  - phân tích false positives / false negatives
"""
))


## 7) Correlation Analysis

Với fraud dataset có hàng trăm feature, heatmap full-size thường rất khó đọc.  
Ở đây notebook dùng 2 cách trực quan hơn:

1. Heatmap cho **tập feature được chọn lọc**
2. Bảng **top correlated pairs** để tìm các cụm biến tương quan mạnh

Lưu ý: correlation cao **không đồng nghĩa** feature vô ích.  
Nhưng nó rất quan trọng cho:

- feature redundancy
- multicollinearity
- cách regularize / select feature
- interpretability của model


In [ ]:

heatmap_feature_candidates = dedupe_keep_order(
    [c for c in ["TransactionAmt", "TransactionDT", "card1", "card2", "addr1", "dist1", "D1", "D4", "D10", "D15", "id_01", "id_02"] if c in df.columns]
    + (numeric_separation["feature"].head(MAX_HEATMAP_FEATURES).tolist() if not numeric_separation.empty else [])
)

heatmap_features = [
    c for c in heatmap_feature_candidates
    if c in ranking_sample.columns and pd.api.types.is_numeric_dtype(ranking_sample[c])
][:MAX_HEATMAP_FEATURES]

if heatmap_features:
    corr_heatmap = ranking_sample[heatmap_features].corr()

    plt.figure(figsize=(max(10, 0.6 * len(heatmap_features)), max(8, 0.6 * len(heatmap_features))))
    sns.heatmap(corr_heatmap, cmap="coolwarm", center=0)
    plt.title("Correlation heatmap (selected numerical features)")
    plt.tight_layout()
    plt.show()

def top_correlated_pairs(frame: pd.DataFrame, threshold: float = 0.70, top_n: int = 25) -> pd.DataFrame:
    corr = frame.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_corr"})
        .sort_values("abs_corr", ascending=False)
    )
    return pairs[pairs["abs_corr"] >= threshold].head(top_n)

pair_feature_candidates = dedupe_keep_order(
    (numeric_separation["feature"].head(80).tolist() if not numeric_separation.empty else [])
    + [c for c in ["TransactionAmt", "TransactionDT", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"] if c in df.columns]
)

pair_feature_candidates = [
    c for c in pair_feature_candidates
    if c in ranking_sample.columns
    and pd.api.types.is_numeric_dtype(ranking_sample[c])
    and ranking_sample[c].notna().mean() >= 0.20
    and ranking_sample[c].nunique(dropna=True) > 1
]

corr_pairs = top_correlated_pairs(ranking_sample[pair_feature_candidates], threshold=0.70, top_n=30) if pair_feature_candidates else pd.DataFrame()
display(corr_pairs)

display(Markdown(
    """
### Nhận xét correlation

- Các cặp tương quan rất cao là ứng viên để xem lại tính dư thừa.
- Tuy nhiên với tree-based models như LightGBM/XGBoost, multicollinearity thường ít nghiêm trọng hơn linear models.
- Dù vậy, correlation analysis vẫn rất hữu ích để:
  - hiểu cấu trúc feature space
  - tránh giải thích model sai
  - quyết định nhóm feature nào nên ưu tiên hoặc gom lại ở bước sau
"""
))


## 8) Outlier Analysis

Trong fraud detection, outlier là phần **cần phân tích rất cẩn thận**.

Khác với nhiều bài toán tabular khác, outlier ở đây **không phải lúc nào cũng nên loại bỏ**.  
Nó có thể là:

- giao dịch bất thường thực sự
- tín hiệu rủi ro mạnh
- một pattern rất hữu ích cho Recall
- hoặc một nguồn gây false positive nếu model học quá mạnh từ tail

Vì vậy mục tiêu ở đây là **hiểu outlier**, không phải xoá máy móc.


In [ ]:

def build_outlier_table(frame: pd.DataFrame, target_col: str, positive_label=1, features: list[str] | None = None) -> pd.DataFrame:
    rows = []
    if features is None:
        features = [
            c for c in frame.select_dtypes(include=[np.number]).columns
            if c not in {target_col, "TransactionID"} and frame[c].nunique(dropna=True) > 5
        ]

    for col in features:
        valid = frame[[col, target_col]].dropna()
        if valid.empty:
            continue

        q1, q3 = valid[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_mask = (valid[col] < lower) | (valid[col] > upper)

        fraud_mask = valid[target_col] == positive_label
        non_fraud_mask = valid[target_col] != positive_label

        rows.append({
            "feature": col,
            "lower_bound": float(lower),
            "upper_bound": float(upper),
            "overall_outlier_pct": float(outlier_mask.mean() * 100),
            "fraud_outlier_pct": float(outlier_mask[fraud_mask].mean() * 100) if fraud_mask.any() else np.nan,
            "non_fraud_outlier_pct": float(outlier_mask[non_fraud_mask].mean() * 100) if non_fraud_mask.any() else np.nan,
            "outlier_pct_gap": (
                float(outlier_mask[fraud_mask].mean() * 100) - float(outlier_mask[non_fraud_mask].mean() * 100)
            ) if fraud_mask.any() and non_fraud_mask.any() else np.nan,
        })

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    return result.sort_values(["outlier_pct_gap", "overall_outlier_pct"], ascending=False)

outlier_features = dedupe_keep_order(
    [c for c in ["TransactionAmt", "dist1", "dist2", "D1", "D4", "D10", "D15"] if c in df.columns]
    + (numeric_separation["feature"].head(10).tolist() if not numeric_separation.empty else [])
)[:10]

outlier_table = build_outlier_table(
    ranking_sample,
    TARGET_COL,
    positive_label=POSITIVE_LABEL,
    features=[c for c in outlier_features if c in ranking_sample.columns]
)

display(outlier_table)

outlier_plot_features = outlier_table["feature"].head(6).tolist() if not outlier_table.empty else []

for col in outlier_plot_features:
    plot_df = plot_sample[[col, TARGET_COL]].dropna().copy()

    q01, q99 = plot_df[col].quantile([0.01, 0.99])
    clipped_df = plot_df[plot_df[col].between(q01, q99)]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    non_fraud_series = plot_df.loc[plot_df[TARGET_COL] != POSITIVE_LABEL, col]
    fraud_series = plot_df.loc[plot_df[TARGET_COL] == POSITIVE_LABEL, col]

    axes[0].boxplot([non_fraud_series, fraud_series], labels=["Non-Fraud", "Fraud"])
    axes[0].set_title(f"Boxplot by class - {col}")
    axes[0].set_ylabel(col)

    nf_clip = clipped_df.loc[clipped_df[TARGET_COL] != POSITIVE_LABEL, col]
    fr_clip = clipped_df.loc[clipped_df[TARGET_COL] == POSITIVE_LABEL, col]
    axes[1].hist(nf_clip, bins=40, density=True, histtype="step", label="Non-Fraud")
    axes[1].hist(fr_clip, bins=40, density=True, histtype="step", label="Fraud")
    axes[1].set_title(f"Body distribution clipped to 1%-99% - {col}")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Density")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

display(Markdown(
    """
### Nhận xét outlier analysis

- Nếu fraud class có tỷ lệ outlier cao hơn rõ rệt ở một feature, đó là tín hiệu rất đáng chú ý.
- Tuy nhiên không nên quyết định `winsorize`, `clip`, hay `remove outliers` chỉ dựa trên thói quen xử lý dữ liệu chung.
- Với fraud detection, việc giữ lại outlier có thể giúp tăng **Recall**, nhưng cũng có thể làm tăng **false positives**, ảnh hưởng Precision.  
  Đây là trade-off phải kiểm chứng ở bước modeling và threshold tuning.
"""
))


## 9) Time-based Analysis

Nếu dataset có biến thời gian, đây thường là một nguồn insight rất mạnh cho fraud detection.

Notebook sẽ kiểm tra tự động:

- nếu có `TransactionDT`: xem như **relative time**
- nếu có datetime/timestamp khác: cố parse về datetime
- nếu không có cột thời gian: ghi rõ trong notebook

> Với IEEE-CIS style data, `TransactionDT` thường được dùng như **elapsed seconds from a reference point** chứ không phải calendar timestamp tuyệt đối.  
> Vì vậy phần này là **relative-time analysis**, không phải phân tích theo ngày-tháng thực tế.


In [ ]:

TIME_COL = None
for candidate in ["TransactionDT", "transaction_time", "timestamp", "datetime", "event_time", "Time", "time"]:
    if candidate in df.columns:
        TIME_COL = candidate
        break

time_summary_md = ""

if TIME_COL is None:
    time_summary_md = """
### Kết luận time-based analysis

Không phát hiện cột thời gian trong dataset đang load, nên bỏ qua phân tích theo thời gian.
"""
else:
    if TIME_COL == "TransactionDT" and pd.api.types.is_numeric_dtype(df[TIME_COL]):
        time_df = df[[TIME_COL, TARGET_COL]].dropna().copy()
        time_df["relative_day"] = (time_df[TIME_COL] // 86400).astype(int)
        time_df["hour_of_day"] = ((time_df[TIME_COL] // 3600) % 24).astype(int)

        daily = time_df.groupby("relative_day")[TARGET_COL].agg(
            transaction_count="size",
            fraud_count=lambda s: (s == POSITIVE_LABEL).sum()
        )
        daily["fraud_rate"] = daily["fraud_count"] / daily["transaction_count"]

        hourly = time_df.groupby("hour_of_day")[TARGET_COL].agg(
            transaction_count="size",
            fraud_count=lambda s: (s == POSITIVE_LABEL).sum()
        )
        hourly["fraud_rate"] = hourly["fraud_count"] / hourly["transaction_count"]

        fig, axes = plt.subplots(3, 1, figsize=(15, 12))

        axes[0].plot(daily.index, daily["transaction_count"])
        axes[0].set_title("Transactions by relative day")
        axes[0].set_xlabel("Relative day index")
        axes[0].set_ylabel("Transaction count")

        axes[1].plot(daily.index, daily["fraud_count"])
        axes[1].set_title("Fraud count by relative day")
        axes[1].set_xlabel("Relative day index")
        axes[1].set_ylabel("Fraud count")

        axes[2].plot(daily.index, daily["fraud_rate"])
        axes[2].set_title("Fraud rate by relative day")
        axes[2].set_xlabel("Relative day index")
        axes[2].set_ylabel("Fraud rate")

        plt.tight_layout()
        plt.show()

        fig, axes = plt.subplots(1, 2, figsize=(15, 4))

        axes[0].plot(hourly.index, hourly["transaction_count"])
        axes[0].set_title("Transactions by hour-of-day (relative)")
        axes[0].set_xlabel("Hour of day")
        axes[0].set_ylabel("Transaction count")

        axes[1].plot(hourly.index, hourly["fraud_rate"])
        axes[1].set_title("Fraud rate by hour-of-day (relative)")
        axes[1].set_xlabel("Hour of day")
        axes[1].set_ylabel("Fraud rate")

        plt.tight_layout()
        plt.show()

        peak_hour = int(hourly["fraud_rate"].idxmax())
        peak_day = int(daily["fraud_rate"].idxmax())

        time_summary_md = f"""
### Kết luận time-based analysis

- Cột thời gian dùng cho phân tích: **`{TIME_COL}`**
- Dữ liệu được phân tích như **relative time**, không phải calendar timestamp tuyệt đối.
- Hour có fraud rate cao nhất (theo relative hour-of-day): **{peak_hour}**
- Relative day có fraud rate cao nhất: **{peak_day}**

Nếu pattern fraud rate thay đổi mạnh theo time window, đây là tín hiệu rất đáng cân nhắc cho bước modeling sau, đặc biệt cho:
- temporal validation
- leakage checking
- threshold adjustment theo context thời gian
"""
    else:
        parsed_time = pd.to_datetime(df[TIME_COL], errors="coerce")
        valid_mask = parsed_time.notna()

        if valid_mask.sum() == 0:
            time_summary_md = f"""
### Kết luận time-based analysis

Có cột `{TIME_COL}` nhưng không parse được thành datetime đủ tốt để phân tích.
"""
        else:
            time_df = pd.DataFrame({
                "time": parsed_time[valid_mask],
                TARGET_COL: df.loc[valid_mask, TARGET_COL].values
            })
            time_df["date"] = time_df["time"].dt.date
            time_df["hour"] = time_df["time"].dt.hour

            daily = time_df.groupby("date")[TARGET_COL].agg(
                transaction_count="size",
                fraud_count=lambda s: (s == POSITIVE_LABEL).sum()
            )
            daily["fraud_rate"] = daily["fraud_count"] / daily["transaction_count"]

            hourly = time_df.groupby("hour")[TARGET_COL].agg(
                transaction_count="size",
                fraud_count=lambda s: (s == POSITIVE_LABEL).sum()
            )
            hourly["fraud_rate"] = hourly["fraud_count"] / hourly["transaction_count"]

            fig, axes = plt.subplots(2, 1, figsize=(15, 8))
            axes[0].plot(daily.index, daily["transaction_count"])
            axes[0].set_title("Transactions by date")
            axes[0].set_xlabel("Date")
            axes[0].set_ylabel("Transaction count")

            axes[1].plot(daily.index, daily["fraud_rate"])
            axes[1].set_title("Fraud rate by date")
            axes[1].set_xlabel("Date")
            axes[1].set_ylabel("Fraud rate")
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(12, 4))
            plt.plot(hourly.index, hourly["fraud_rate"])
            plt.title("Fraud rate by hour")
            plt.xlabel("Hour")
            plt.ylabel("Fraud rate")
            plt.tight_layout()
            plt.show()

            time_summary_md = f"""
### Kết luận time-based analysis

- Cột thời gian dùng cho phân tích: **`{TIME_COL}`**
- Đã parse thành datetime thành công.
- Cần kiểm tra xem fraud rate có drift theo thời gian hay không trước khi chốt strategy validation.
"""

display(Markdown(time_summary_md))


## 10) EDA Summary

Phần này tổng hợp các insight quan trọng nhất từ EDA và liên hệ trực tiếp tới bước modeling sau này.


In [ ]:

eda_summary_lines = []

eda_summary_lines.append(f"- Fraud ratio hiện tại: **{fraud_ratio:.4%}**. Đây là dấu hiệu rõ ràng rằng cần ưu tiên **Recall, Precision, F1-score và PR-AUC** hơn accuracy.")

if high_missing_cols:
    eda_summary_lines.append(f"- Có **{len(high_missing_cols)} cột** có missing >= 90%. Không nên drop tự động toàn bộ trước khi kiểm tra xem missingness có mang tín hiệu fraud hay không.")

eda_summary_lines.append(f"- Duplicate rows: **{duplicate_rows:,}**; duplicate `TransactionID`: **{duplicate_transaction_ids:,}**.")

if not numeric_separation.empty:
    top_numeric = ', '.join(f'`{c}`' for c in numeric_separation['feature'].head(8).tolist())
    eda_summary_lines.append(f"- Các numerical features nổi bật về khả năng tách class ở mức EDA: {top_numeric}.")

if not categorical_fraud_rate_table.empty:
    top_cat_rows = categorical_fraud_rate_table.head(5)
    top_cat_signals = []
    for _, row in top_cat_rows.iterrows():
        top_cat_signals.append(f"`{row['feature']}={row['category']}` ({row['fraud_rate_pct']:.2f}%)")
    eda_summary_lines.append("- Một số categorical signals đáng chú ý theo fraud rate: " + ", ".join(top_cat_signals) + ".")

if not corr_pairs.empty:
    pair_preview = ", ".join(
        f"`{row['feature_1']}`–`{row['feature_2']}` ({row['abs_corr']:.2f})"
        for _, row in corr_pairs.head(5).iterrows()
    )
    eda_summary_lines.append(f"- Có những cặp feature tương quan mạnh cần lưu ý khi giải thích model hoặc giảm dư thừa: {pair_preview}.")

if not outlier_table.empty:
    outlier_preview = ", ".join(
        f"`{row['feature']}` (fraud outlier cao hơn {row['outlier_pct_gap']:.2f} điểm %)"
        for _, row in outlier_table.head(5).iterrows()
    )
    eda_summary_lines.append(f"- Outlier có vẻ mang tín hiệu cho fraud ở một số feature: {outlier_preview}.")

if TIME_COL is not None:
    eda_summary_lines.append(f"- Dataset có cột thời gian **`{TIME_COL}`** nên bước modeling sau cần cân nhắc temporal validation hoặc ít nhất phải kiểm tra drift theo time window.")

eda_summary_lines.extend([
    "- Với fraud detection, mô hình baseline nên bắt đầu bằng **stratified split**, ưu tiên đọc **PR curve** và tune threshold theo business objective thay vì dùng mặc định 0.5.",
    "- Nếu mục tiêu ngắn hạn là **không bỏ sót fraud**, cần tối ưu Recall nhưng đồng thời theo dõi Precision để tránh số lượng false positives quá lớn.",
    "- Các insight EDA ở đây chưa phải feature engineering chính thức, nhưng đã chỉ ra những nhóm feature và pattern quan trọng để bước modeling khai thác tiếp."
])

display(Markdown("### Tóm tắt insight chính\n\n" + "\n".join(eda_summary_lines)))


## 11) Đánh giá DVC cho repository hiện tại

Phần này **không trả lời lý thuyết chung chung** mà bám sát repo hiện tại.

Mục tiêu là trả lời rõ một trong ba hướng:

1. Nên dùng DVC ngay  
2. Nên làm EDA trước, DVC sau  
3. Chưa cần DVC ở giai đoạn này


In [ ]:

def safe_path_size_mb(path: Path) -> float:
    if path.is_file():
        return path.stat().st_size / (1024 ** 2)
    return np.nan

top_level_entries = []
for path in sorted(repo_root.iterdir(), key=lambda p: (p.is_file(), p.name.lower())):
    if path.name.startswith(".") and path.name not in {".dvc"}:
        continue
    top_level_entries.append({
        "name": path.name,
        "type": "dir" if path.is_dir() else "file",
        "size_mb_if_file": safe_path_size_mb(path),
    })

repo_snapshot_df = pd.DataFrame(top_level_entries)
display(repo_snapshot_df)

raw_data_audit = pd.DataFrame({
    "file_name": [p.name for p in sorted(raw_data_dir.iterdir()) if p.is_file()],
    "size_mb": [p.stat().st_size / (1024 ** 2) for p in sorted(raw_data_dir.iterdir()) if p.is_file()],
}).sort_values("size_mb", ascending=False)

display(raw_data_audit)
print(f"Total raw_data size: {raw_data_audit['size_mb'].sum():,.2f} MB")


### Kết luận DVC cho repo này: **Nên làm EDA trước, DVC sau**

Đây là quyết định thực tế nhất cho repo hiện tại.

### Vì sao không chọn “DVC ngay lập tức trước mọi thứ”?

Vì ở snapshot hiện tại, project của bạn mới ở giai đoạn rất sớm:

- repo còn khá tối giản
- trọng tâm hiện tại là **EDA**
- reusable pipeline code / model training pipeline chưa hình thành rõ
- chưa có processed data, model artifacts, metrics history, experiment outputs ở quy mô khiến Git workflow trở nên rối ngay lập tức

Nếu ép triển khai DVC quá sớm trước khi có notebook EDA ổn định và trước khi repo có cấu trúc tốt hơn, nhóm rất dễ:

- tăng overhead công cụ
- dùng DVC nửa vời
- commit lẫn lộn giữa Git và DVC
- mất thời gian vận hành hơn là tăng chất lượng project

### Nhưng vì sao cũng không nên kết luận “chưa cần DVC”?

Vì dữ liệu hiện tại **đã đủ lớn** để DVC bắt đầu có ích ngay sau khi EDA ổn định:

- raw dataset nhiều file
- train/test transaction ở dạng file lớn
- sau EDA gần như chắc chắn sẽ phát sinh:
  - processed datasets
  - selected feature sets
  - saved models
  - predictions
  - evaluation metrics
  - plots / reports

Nếu tiếp tục để toàn bộ dữ liệu và artifacts lớn đi theo Git thường, repo sẽ nhanh chóng khó quản lý, khó sync giữa máy, và khó tái lập.

### Kết luận thực dụng

**Thứ tự hợp lý cho project sinh viên này là:**

1. Hoàn tất notebook EDA hiện tại  
2. Commit notebook + cấu trúc repo sạch bằng Git  
3. Sau đó tích hợp DVC ở mức tối thiểu để quản lý `raw_data/` trước khi project sinh thêm processed data và model artifacts


In [ ]:

tracking_plan = pd.DataFrame([
    {
        "artifact_group": "README.md, requirements.txt, params.yaml, .gitignore",
        "track_with": "Git",
        "reason": "File text/code/config nhỏ, cần review qua Git"
    },
    {
        "artifact_group": "notebooks/*.ipynb, src/**/*.py",
        "track_with": "Git",
        "reason": "Notebook và source code cần diff, review và version cùng code"
    },
    {
        "artifact_group": "raw_data/",
        "track_with": "DVC",
        "reason": "Data thô tương đối lớn, ít phù hợp để giữ trực tiếp trong Git"
    },
    {
        "artifact_group": "data/processed/, artifacts/models/, predictions lớn",
        "track_with": "DVC",
        "reason": "Sinh ra từ pipeline và có thể thay đổi theo experiment"
    },
    {
        "artifact_group": "metrics JSON nhỏ, selected PNG nhỏ",
        "track_with": "Git hoặc DVC tùy quy mô",
        "reason": "Nếu ít và nhỏ có thể Git; nếu nhiều run/phiên bản thì chuyển sang DVC"
    },
    {
        "artifact_group": ".dvc/config",
        "track_with": "Git",
        "reason": "Project-level config nên chia sẻ cho cả nhóm"
    },
    {
        "artifact_group": ".dvc/config.local, OAuth tokens, credential files",
        "track_with": "Không commit",
        "reason": "Chứa thông tin local/secret, phải giữ ngoài Git"
    },
])

display(tracking_plan)


## 12) Cách triển khai DVC tối thiểu nhưng đủ dùng cho project sinh viên

Mức triển khai hợp lý ở giai đoạn kế tiếp là:

- **chưa cần pipeline DVC đầy đủ ngay**
- **chưa cần experiment tracking phức tạp ngay**
- trước hết chỉ cần:
  - init DVC
  - đưa `raw_data/` ra khỏi Git thường
  - cấu hình 1 remote chung cho nhóm
  - dùng `dvc push/pull` để đồng bộ dữ liệu
  - giữ notebook / code / config bằng Git

### Cấu trúc thư mục tối thiểu đề xuất

```text
Fraud-Detection/
├── raw_data/                      # DVC track
├── data/
│   └── processed/                # DVC track khi bắt đầu preprocessing
├── notebooks/
│   └── 01_fraud_eda_and_dvc.ipynb
├── src/
│   ├── data/
│   ├── features/
│   ├── models/
│   └── utils/
├── artifacts/
│   ├── figures/
│   ├── metrics/
│   ├── models/
│   └── predictions/
├── README.md
├── requirements.txt
├── params.yaml                   # thêm khi bắt đầu training
├── dvc.yaml                      # thêm khi bắt đầu stage pipeline
└── dvc.lock                      # sinh ra sau khi có pipeline
```

### Remote storage phù hợp cho nhóm sinh viên

**Khuyến nghị chính:** dùng **Google Drive shared folder** nếu nhóm cần cách thiết lập rẻ và dễ chia sẻ nhất.

Lý do:

- đa số nhóm sinh viên đã dùng Google Drive
- không cần thuê hạ tầng riêng
- đủ tốt cho mức đồng bộ dataset và artifacts cơ bản

**Fallback đơn giản hơn** nếu nhóm chủ yếu làm trên 1 máy hoặc 1 ổ đĩa chung:
- dùng **local filesystem remote** như một thư mục ngoài repo

> Nếu về sau nhóm làm CI/CD hoặc cần automation nghiêm túc hơn, có thể chuyển từ Google Drive sang GCS / S3.


### 12.1) Cài đặt DVC

Chạy trong terminal ở root repo:

```bash
# trong môi trường Python riêng
pip install dvc

# nếu dùng Google Drive làm remote
pip install "dvc[gdrive]"
```

### 12.2) Init DVC

```bash
git status
dvc init
git add .dvc .gitignore
git commit -m "Initialize DVC"
```

### 12.3) Đưa dataset vào DVC

Với repo hiện tại, bước tối thiểu hợp lý là track **toàn bộ thư mục `raw_data/`** bằng DVC:

```bash
dvc add raw_data
git add raw_data.dvc .gitignore
git commit -m "Track raw dataset with DVC"
```

> Sau lệnh này, dữ liệu trong `raw_data/` sẽ được DVC theo dõi bằng file metadata `raw_data.dvc`, còn Git sẽ track metadata thay vì track trực tiếp dữ liệu lớn.


### 12.4) Cấu hình remote cho nhóm

#### Option A — Google Drive shared folder (khuyến nghị cho nhóm sinh viên)

Tạo một folder chung trên Google Drive, share cho tất cả thành viên, rồi dùng **folder ID**:

```bash
dvc remote add -d storage gdrive://<FOLDER_ID>/Fraud-Detection
dvc remote modify storage gdrive_acknowledge_abuse true
git add .dvc/config
git commit -m "Configure DVC Google Drive remote"
```

Lần đầu `dvc push` hoặc `dvc pull`, DVC sẽ yêu cầu xác thực Google account có quyền truy cập folder đó.

```bash
dvc push
```

Mỗi thành viên khác sau khi clone repo:

```bash
git pull
dvc pull
```

#### Option B — Local filesystem remote (nếu muốn tối giản hơn nữa)

Ví dụ dùng một thư mục ngoài repo:

```bash
mkdir -p ../dvc_storage/fraud-detection
dvc remote add -d storage ../dvc_storage/fraud-detection
git add .dvc/config
git commit -m "Configure local DVC remote"
dvc push
```

Option này phù hợp nếu nhóm chưa cần chia sẻ remote qua Internet hoặc chỉ làm trên 1 máy / 1 ổ đĩa chung.


### 12.5) File nào vào Git, file nào vào DVC?

#### Commit bằng Git thường

```text
README.md
requirements.txt
notebooks/*.ipynb
src/**/*.py
params.yaml
dvc.yaml
dvc.lock
.dvc/config
raw_data.dvc
```

#### Track bằng DVC

```text
raw_data/
data/processed/
artifacts/models/
artifacts/predictions/
artifact files lớn / nhiều phiên bản
```

#### Không commit

```text
.dvc/config.local
token đăng nhập / OAuth cached credentials
service account keys
bất kỳ secrets local nào
```


### 12.6) Workflow tối thiểu hằng ngày cho nhóm

```bash
# nhận code + metadata mới
git pull

# đồng bộ dữ liệu lớn từ remote
dvc pull

# sau khi cập nhật raw_data hoặc processed data
dvc add raw_data
git add raw_data.dvc .gitignore
git commit -m "Update raw dataset version"
dvc push
git push
```

Nếu dữ liệu không đổi mà chỉ đổi notebook / code:

```bash
git add notebooks/ src/ README.md
git commit -m "Update EDA notebook"
git push
```


### 12.7) Khi nào nên chuyển từ `dvc add` sang `dvc.yaml` pipeline?

Ngay bây giờ, **chưa bắt buộc** phải có pipeline DVC đầy đủ.

Nhưng khi project bắt đầu có quy trình kiểu:

- raw data → cleaned data
- cleaned data → train/valid split
- split → feature matrix
- feature matrix → model
- model → metrics / predictions

thì nên chuyển sang dùng **DVC stages** để tái lập pipeline.

Ví dụ tối thiểu sau này:

```bash
dvc stage add -n preprocess \
  -d src/data/preprocess.py \
  -d raw_data \
  -o data/processed \
  python src/data/preprocess.py

dvc stage add -n train_baseline \
  -d src/models/train_baseline.py \
  -d data/processed \
  -p train.seed,train.test_size,train.model_name \
  -o artifacts/models/baseline_model.joblib \
  -M artifacts/metrics/baseline_metrics.json \
  python src/models/train_baseline.py
```

Lúc đó `dvc.yaml` + `dvc.lock` sẽ giúp project đi đúng hướng MLOps hơn:
- tái lập
- rõ dependency / output
- dễ so sánh experiment
- rõ file nào sinh ra từ bước nào


## 13) Manual edits checklist

Bạn nên rà lại 3 chỗ sau trước khi chạy thật:

1. **Đường dẫn dữ liệu**
   - Notebook đang giả định dữ liệu nằm ở `raw_data/`
   - Nếu bạn đổi thư mục, sửa phần `find_repo_root()` hoặc các biến path

2. **Tên target**
   - Notebook đang tự dò `isFraud`, `fraud`, `target`, `Class`, `class`, `label`
   - Nếu target khác, set `TARGET_COL` thủ công

3. **Cột thời gian**
   - Notebook ưu tiên `TransactionDT`
   - Nếu dataset thực tế dùng tên khác, thêm vào danh sách candidate

4. **Cấu trúc feature khác chuẩn IEEE-CIS**
   - Dtype map hiện tại được tối ưu cho schema kiểu IEEE-CIS
   - Nếu dataset thực tế khác nhiều, chỉ cần chỉnh lại hàm `build_ieee_dtype_map()`


## 14) Final recommendation cho project hiện tại

**Quyết định cuối cùng:** **Nên làm EDA trước, DVC sau**.

Cụ thể:

- **Ngay bây giờ**: tập trung hoàn tất EDA notebook này và commit bằng Git
- **Ngay sau đó**: tích hợp DVC ở mức tối thiểu để track `raw_data/`
- **Khi bắt đầu preprocessing / training có cấu trúc**: thêm `dvc.yaml` stages

Đây là phương án cân bằng nhất giữa:
- tính thực dụng
- độ sạch của repo
- khả năng cộng tác nhóm
- hướng phát triển MLOps cho project sinh viên
